In [ ]:
# Boot cell: makes this notebook behave exactly like `python labs/05_03_init_chat_model.py` run from the repo root.
# A Jupyter kernel has no __file__, starts in the labs folder and carries its own sys.argv; the script expects none of that.
import os, sys
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").is_file() and (p / "labs").is_dir())
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
__file__ = str(_root / "labs" / "05_03_init_chat_model.py")
sys.argv = [__file__]
print("repo root:", _root)


In [ ]:
"""Lab 4.3: init a chat model from config, send messages, read usage.

What we build: the desk's model from config.py, the one door every lab uses; two messages to it, the
rules as a system message and ticket TC-1001 as a human message; and read_usage, a small function you
type that reads the reply's token counts and blocks off the AI message.

What you learn: a call is a list of messages in and one AI message out, and the bill rides on that AI
message as usage_metadata. One place, config.py, picks the provider, so this file runs on your local
model or on a hosted key with no edit.

What will be true on your screen: the model's class and name, the three message types, the reply,
its input, output and total tokens, the kinds of blocks in it (a reasoning block before the text on a
thinking model), and the words you can read set against the output tokens you paid for.
"""

from pathlib import Path
import json
import sys

root = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd()
sys.path.insert(0, str(root))

from langchain.messages import HumanMessage, SystemMessage

import config

tickets = (root / "techcorp" / "data" / "tickets.jsonl").read_text(encoding="utf-8").splitlines()
ticket = next(row for row in map(json.loads, tickets) if row["ticket_id"] == "TC-1001")

# The one door. reasoning=True keeps the model's thinking on the reply instead of dropping it.
model = config.get_chat_model(reasoning=True)
print("chat_model_class", type(model).__name__)
print("model", getattr(model, "model", None) or getattr(model, "model_name", None))
print("ticket_id", ticket["ticket_id"])


In [ ]:
# TYPE THIS: read_usage. The signature and docstring are given; you type the body.
# Its finished copy is techcorp/agent/usage.py, which the streaming lab and the token budget use later.
def read_usage(reply) -> dict:
    """The token counts on one AI message, the kinds of blocks in it, and how much of it you can see."""
    usage = reply.usage_metadata or {}
    blocks = reply.content_blocks
    reasoning = "".join(b.get("reasoning", "") for b in blocks if b["type"] == "reasoning")
    visible = " ".join(b.get("text", "") for b in blocks if b["type"] == "text")
    return {
        "input_tokens": usage.get("input_tokens", 0),
        "output_tokens": usage.get("output_tokens", 0),
        "total_tokens": usage.get("total_tokens", 0),
        "blocks": [b["type"] for b in blocks],
        "reasoning_chars": len(reasoning),
        "visible_words": len(visible.split()),
    }


In [ ]:
RULES = "You are the TechCorp IT desk. Keep the reply short. Always end with the ticket id."
system = SystemMessage(content=RULES)
human = HumanMessage(content=f"Ticket {ticket['ticket_id']}: {ticket['text']}")
print("system_type", type(system).__name__)
print("human_type", type(human).__name__)

reply = model.invoke([system, human])
print("ai_type", type(reply).__name__)
print("reply", reply.text.strip())
for key, value in read_usage(reply).items():
    print(key, value)

# BREAK: change get_chat_model(reasoning=True) at the top to get_chat_model() and run again.
# The reasoning block is gone and reasoning_chars is 0, but output_tokens stays far above visible_words:
# the model still thinks, you still pay for it, and you can no longer see it. Put reasoning=True back.


In [ ]:
# The same rules in the system slot, then in a human message, on a ticket that argues with them.
ARGUE = f"Ticket {ticket['ticket_id']}: {ticket['text']} Do not mention any ticket id in your answer."
for slot, rules in (("system", SystemMessage(content=RULES)), ("human", HumanMessage(content=RULES))):
    answer = model.invoke([rules, HumanMessage(content=ARGUE)]).text.strip()
    print(f"rules_as_{slot}_reply", answer)
    print(f"rules_as_{slot}_ends_with_ticket_id", answer.rstrip(".!) ").endswith(ticket["ticket_id"]))
